# Basic Usage Examples

This notebook demonstrates basic usage of the ethnicolr package for predicting race and ethnicity from names.

## Setup

First, let's import the necessary libraries and load some sample data.

In [ ]:
import pandas as pd
import ethnicolr
from pathlib import Path

# Load sample data
data_path = Path('data/input-with-header.csv')
df = pd.read_csv(data_path)
print(f"Sample data shape: {df.shape}")
print("\nFirst few rows:")
df.head()

## Census Data Lookup

The simplest approach is to look up demographic probabilities by last name using US Census data.

In [ ]:
# Census lookup by last name (2010 data)
result_census = ethnicolr.census_ln(df, 'last_name', year=2010)
print(f"Result shape: {result_census.shape}")
print("\nColumns added:")
census_cols = [col for col in result_census.columns if col not in df.columns]
print(census_cols)

# Show first few results
result_census[['last_name', 'first_name'] + census_cols].head()

## Simple Census-based Predictions

For more sophisticated predictions, we can use the census-based LSTM model.

In [ ]:
# Predict using census LSTM model
result_pred = ethnicolr.pred_census_ln(df, 'last_name')
print(f"Result shape: {result_pred.shape}")
print("\nPrediction columns:")
pred_cols = [col for col in result_pred.columns if col not in df.columns]
print(pred_cols)

# Show predictions with confidence scores
result_pred[['last_name', 'first_name', 'race', 'white', 'black', 'api', 'hispanic']].head(10)

## Summary Statistics

Let's look at the distribution of predicted races in our sample.

In [ ]:
# Distribution of predicted races
race_dist = result_pred['race'].value_counts()
print("Race distribution:")
for race, count in race_dist.items():
    percentage = (count / len(result_pred)) * 100
    print(f"{race}: {count} ({percentage:.1f}%)")

# Show some examples by race
print("\nExample predictions by race:")
for race in race_dist.index[:3]:
    examples = result_pred[result_pred['race'] == race]['last_name'].head(3).tolist()
    print(f"{race}: {', '.join(examples)}")

## Comparing Methods

Let's compare the census lookup vs. ML prediction for a few names.

In [ ]:
# Create comparison dataframe
# Normalize names to handle case differences between models
result_census_norm = result_census.copy()
result_pred_norm = result_pred.copy()

# Convert to lowercase for consistent merging
result_census_norm['last_name_norm'] = result_census_norm['last_name'].str.lower()
result_pred_norm['last_name_norm'] = result_pred_norm['last_name'].str.lower()

comparison = pd.merge(
    result_census_norm[['last_name', 'last_name_norm', 'pctwhite', 'pctblack', 'pctapi', 'pcthispanic']],
    result_pred_norm[['last_name_norm', 'race', 'white', 'black', 'api', 'hispanic']],
    on='last_name_norm',
    how='inner'
)

# Rename for clarity
comparison.columns = [
    'last_name', 'last_name_norm',
    'census_white', 'census_black', 'census_api', 'census_hispanic',
    'pred_race', 'pred_white', 'pred_black', 'pred_api', 'pred_hispanic'
]

print(f"Merged {len(comparison)} names successfully")
print("Comparison of Census vs ML predictions (first 10 names):")
comparison[['last_name', 'census_white', 'census_black', 'pred_race', 'pred_white', 'pred_black']].head(10)

## Key Differences

- **Census lookup**: Returns population-level probabilities for each race/ethnicity based on surname frequency in census data
- **ML prediction**: Uses neural networks trained on census data to predict the most likely race/ethnicity category
- **Use cases**: Census lookup for aggregate analysis, ML predictions for individual classification